# Capitulo 16 - Graph RAG com Neo4j

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap16_graph_rag.ipynb)

Neste capitulo, construimos grafos de conhecimento com Neo4j e os utilizamos
para RAG baseado em grafos (Graph RAG). Cobrimos desde a criacao do grafo
ate consultas Cypher e busca vetorial sobre o grafo.

---

## Atribuicao e Fonte Original

**Fonte:** [RAG-with-Python-Cookbook](https://github.com/PacktPublishing/RAG-with-Python-Cookbook) - Packt Publishing

**Notebooks fonte:** ch09_graph_rag/9.1_basic_sla_graph.ipynb, 9.2_enrich_company_data.ipynb, 9.3_cypher_queries.ipynb, 9.4_embeddings_vector_search.ipynb

**Adaptado e comentado por Anderson Ejepsen** para o livro *LLM On-Premise: Guia Pratico*.

Todos os creditos aos autores originais sao mantidos.

---

## Instalacao de Dependencias

Execute a celula abaixo para instalar todos os pacotes necessarios.

In [ ]:
# Instalar dependencias do capitulo
%pip install -q python-dotenv neo4j openai pandas requests

---
## Parte 1: Grafo Basico de SLA

Criamos um grafo de conhecimento basico representando contratos SLA (Service Level Agreement).
Esta e a base para entender como modelar dados em grafos.

## Instalar Pacotes Necessarios

Uncomment and run if needed:
```bash
pip install python-dotenv neo4j
```

In [ ]:
%pip install python-dotenv==1.0.0 neo4j==5.0.0

## Import Libraries

In [ ]:
from __future__ import annotations
import argparse
import os
import re
from dataclasses import dataclass
from typing import List
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

## Connect to Neo4j

Read the Neo4j connection details from environment variables and create a Neo4j driver instance.

In [ ]:
## tag::code_neo4j_sla_connection[]
def get_driver():
    """Create Neo4j driver from environment variables."""
    uri = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
    user = os.getenv("NEO4J_USERNAME", "neo4j")
    pwd = os.getenv("NEO4J_PASSWORD", "testpassword")
    return GraphDatabase.driver(uri, auth=(user, pwd))
## end::code_neo4j_sla_connection[]

## Create Database Constraints

Define and create uniqueness constraints for SLA, Clause, and ClauseType nodes. These constraints prevent duplicate nodes and keep MERGE operations predictable.

In [ ]:
## tag::code_neo4j_constraints[]
def create_constraints(driver):
    """Create uniqueness constraints for graph nodes."""
    constraints = [
        "CREATE CONSTRAINT sla_id IF NOT EXISTS " "FOR (s:SLA) REQUIRE s.id IS UNIQUE",
        "CREATE CONSTRAINT clause_id IF NOT EXISTS "
        "FOR (c:Clause) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT type_name IF NOT EXISTS "
        "FOR (t:ClauseType) REQUIRE t.name IS UNIQUE",
    ]
    with driver.session() as session:
        for constraint in constraints:
            session.run(constraint)


driver = get_driver()
create_constraints(driver)
print("✓ Constraints created")
## end::code_neo4j_constraints[]

## Define Clause Data Structure and Parser

Parse the SLA markdown file into Clause objects. Each '##' heading is treated as a clause title and the following lines as the clause text.

In [ ]:
# tag::code_sla_chunking[]
@dataclass
class Clause:
    id: str
    title: str
    text: str
    order: int
    clause_type: str = "Other"


def parse_sla_file(path: str, sla_id: str) -> List[Clause]:
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    sections = re.split(r"^##\s+", content, flags=re.MULTILINE)[1:]
    clauses = []

    for idx, section in enumerate(sections, start=1):
        lines = section.strip().splitlines()
        if not lines:
            continue
        title = lines[0].strip()
        text = "\n".join(lines[1:]).strip()

        clauses.append(Clause(id=f"{sla_id}_C{idx}", title=title, text=text, order=idx))

    return clauses
# end::code_sla_chunking[]

## Parse SLA File

Load and parse the SLA document.

In [ ]:
sla_id = "SLA1"
sla_title = "Cloud Compute Service Level Agreement (SLA)"
sla_file_path = "../datasets/sample_data_graph_rag/SLA1_SUP1.md"

clauses = parse_sla_file(path=sla_file_path, sla_id=sla_id)
print(f"✓ Parsed {len(clauses)} clauses from {sla_file_path}")

## Write SLA and Clauses to Neo4j

This function writes the SLA and all Clause nodes to Neo4j:
- Creates the SLA node
- Creates Clause nodes and HAS_CLAUSE relationships from the SLA
- Links clauses in order using NEXT relationships

In [ ]:
## tag::write_to_neo4j[]
def write_sla_and_clauses(
    driver, sla_id: str, sla_title: str, clauses: List[Clause]
) -> None:
    with driver.session() as session:
        # SLA node
        session.run(
            """
            MERGE (s:SLA {id: $id})
            SET s.title = $title
            """,
            id=sla_id,
            title=sla_title,
        )

        # Clause nodes + HAS_CLAUSE
        for c in clauses:
            session.run(
                """
                MERGE (cl:Clause {id: $id})
                SET cl.title = $title,
                    cl.text = $text,
                    cl.order = $order
                """,
                id=c.id,
                title=c.title,
                text=c.text,
                order=c.order,
                ctype=c.clause_type,
            )
            session.run(
                """
                MATCH (s:SLA {id: $sla_id})
                MATCH (cl:Clause {id: $cid})
                MERGE (s)-[:HAS_CLAUSE]->(cl)
                """,
                sla_id=sla_id,
                cid=c.id,
            )

        # NEXT relationships to preserve order
        for prev, nxt in zip(clauses, clauses[1:]):
            session.run(
                """
                MATCH (a:Clause {id: $p})
                MATCH (b:Clause {id: $n})
                MERGE (a)-[:NEXT]->(b)
                """,
                p=prev.id,
                n=nxt.id,
            )


write_sla_and_clauses(driver, sla_id, sla_title, clauses)
print("✓ SLA and clauses written to Neo4j")
## end::write_to_neo4j[]

## Infer Clause Types

Add ClauseType to categorize clauses semantically. Use a simple heuristic to map titles like "Availability" or "Support" to semantic types.

In [ ]:
## tag::find_clause_type[]
def infer_clause_type(title: str) -> str:
    """Infer ClauseType based on keywords in the title."""
    title_lower = title.lower()
    keywords = {
        "Availability": ["availability", "uptime"],
        "Support": ["support", "response time", "incident"],
        "Maintenance": ["maintenance"],
        "DataProtection": ["data protection", "gdpr", "privacy"],
        "Liability": ["liability"],
        "Termination": ["termination"],
    }

    for clause_type, words in keywords.items():
        if any(word in title_lower for word in words):
            return clause_type
    return "Other"


for c in clauses:
    c.clause_type = infer_clause_type(c.title)
    
print("✓ Clause types inferred")
## end::find_clause_type[]

## Add ClauseType Nodes and Relationships

Create ClauseType nodes and connect each clause to its type using OF_TYPE relationships.

In [ ]:
## tag::add_clause_types[]
def add_clause_types(session, clauses: List[Clause]):
    # Criar ClauseType nodes
    types = [{"clause_type": c.clause_type} for c in clauses]
    session.run(
        """
        UNWIND $rows AS row
        MERGE (t:ClauseType {name: row.clause_type})
    """,
        rows=types,
    )

    # Link clauses to their types
    links = [{"id": c.id, "type": c.clause_type} for c in clauses]
    session.run(
        """
        UNWIND $rows AS row
        MATCH (cl:Clause {id: row.id})
        MATCH (t:ClauseType {name: row.type})
        MERGE (cl)-[:OF_TYPE]->(t)
    """,
        rows=links,
    )


add_clause_types(driver.session(), clauses)
print("✓ ClauseType nodes and relationships added")
## end::add_clause_types[]

## Verify the Graph

Run some Cypher queries to verify that the nodes and relationships were created correctly.

In [ ]:
with driver.session() as session:
    sla_result = session.run("MATCH (s:SLA) RETURN s.id AS id, s.title AS title")
    print("SLAs in the graph:")
    for record in sla_result:
        print(f"- {record['id']}: {record['title']}")

    clause_result = session.run(
        """
        MATCH (s:SLA)-[:HAS_CLAUSE]->(c:Clause)
        RETURN c.id AS id, c.title AS title, c.clause_type AS type
        ORDER BY c.order
        """
    )
    print("\nClauses in the graph:")
    for record in clause_result:
        print(f"- {record['id']}: {record['title']} (Type: {record['type']})")

## Close Connection

In [ ]:
driver.close()
print("✓ Connection closed")

---
## Parte 2: Enriquecimento de Dados no Grafo

Demonstramos como enriquecer o grafo de conhecimento com dados adicionais de empresas,
utilizando APIs externas e integracao com LLMs.

# Recipe 2: Enrich Company Data in Knowledge Graph

This notebook demonstrates how to build an enriched knowledge graph in Neo4j by loading company information, addresses, spend data, and SLA metadata. You'll learn how to create nodes for different entities and establish relationships between them to create a comprehensive business data graph.

## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [ ]:
%pip install pandas==2.0.0 python-dotenv==1.0.0 neo4j==5.0.0 openai==1.12.0

## 1. Setup and Imports

Import required libraries and load environment variables.

In [ ]:
from __future__ import annotations
import os
from typing import Any, Dict, Optional
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Carregar environment variables from .env file
load_dotenv()

## 2. Connect to Neo4j

Create a reusable driver function that reads connection details from environment variables.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
```

In [ ]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    pwd = os.getenv("NEO4J_PASSWORD")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Inicializar driver
driver = get_driver()
print("✓ Connected to Neo4j")

## 3. Load Companies

Import company information and create `Company` nodes with attributes such as name, country, and industry.

In [ ]:
def load_companies(driver, companies_csv_path: str) -> None:
    """Load companies from CSV and create Company nodes."""
    df = pd.read_csv(companies_csv_path)
    print(f"Loading {len(df)} companies...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MERGE (c:Company {supplier_id: row.supplier_id})
            ON CREATE SET c.name=row.name, c.country=row.country, c.industry=row.industry
            SET c.industry = row.industry
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Companies loaded")

# Execute
companies_csv_path = "../datasets/sample_data_graph_rag/companies.csv"
load_companies(driver, companies_csv_path)

## 4. Load Addresses

Create `Address` nodes with street, city, postal code, and country information.

In [ ]:
def load_addresses(driver, addresses_csv_path: str) -> None:
    """Load addresses from CSV and create Address nodes."""
    df = pd.read_csv(addresses_csv_path)
    print(f"Loading {len(df)} addresses...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MERGE (a:Address {id: row.address_id})
            ON CREATE SET a.street = row.street, 
                          a.city = row.city, 
                          a.postal_code = row.postal_code, 
                          a.country = row.country
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Addresses loaded")

# Execute
addresses_csv_path = "../datasets/sample_data_graph_rag/addresses.csv"
load_addresses(driver, addresses_csv_path)

## 5. Connect Companies and Addresses

Link companies to their addresses with `LOCATED_AT` relationships.

In [ ]:
def connect_company_addresses(driver, companies_csv_path: str) -> None:
    """Create LOCATED_AT relationships between companies and addresses."""
    df = pd.read_csv(companies_csv_path)[["supplier_id", "address_id"]]
    print(f"Connecting {len(df)} company-address relationships...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MATCH (c:Company {supplier_id: row.supplier_id})
            MATCH (a:Address {id: row.address_id})
            MERGE (c)-[:LOCATED_AT]->(a)
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Company-address relationships created")

# Execute
companies_csv_path = "../datasets/sample_data_graph_rag/companies.csv"
connect_company_addresses(driver, companies_csv_path)

## 6. Add Spend Information

Attach yearly spend data to `Company` nodes as properties.

In [ ]:
def load_spend(driver, spend_csv_path: str) -> None:
    """Load spend data and update Company nodes with spend information."""
    df = pd.read_csv(spend_csv_path)
    print(f"Loading spend data for {len(df)} companies...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MATCH (c:Company {supplier_id: row.supplier_id})
            SET c.spend_2024 = toFloat(row.spend_eur), 
                c.spend_category = row.spend_category
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ Spend data loaded")

# Execute
spend_csv_path = "../datasets/sample_data_graph_rag/spend_2024.csv"
load_spend(driver, spend_csv_path)

## 7. Link Suppliers to SLAs

Import SLA metadata and create `HAS_SLA` relationships connecting suppliers to their contracts.

In [ ]:
def load_slas(driver, sla_csv_path: str) -> None:
    """Load SLA metadata and link to Company nodes."""
    df = pd.read_csv(sla_csv_path)
    df["effective_date"] = df["effective_date"].replace({"N/A": None})
    df["effective_date"] = df["effective_date"].where(
        df["effective_date"].notna(), None
    )
    print(f"Loading {len(df)} SLA records...")
    
    with driver.session() as session:
        session.run(
            """
            UNWIND $rows AS row
            MATCH (c:Company {supplier_id: row.supplier_id})
            MERGE (s:SLA {id: row.sla_id})
            ON CREATE SET s.title = row.title, 
                          s.service_name = row.service_name,
                          s.governing_law = row.governing_law,
                          s.effective_date = CASE 
                            WHEN row.effective_date IS NOT NULL 
                            THEN date(row.effective_date) 
                            ELSE NULL END
            MERGE (c)-[:HAS_SLA]->(s)
            """,
            {"rows": df.to_dict(orient="records")},
        )
    print("✓ SLA data loaded and linked to companies")

# Execute
sla_csv_path = "../datasets/sample_data_graph_rag/slas.csv"
load_slas(driver, sla_csv_path)

## 8. Verify the Graph Structure

Run a query to verify that the data was loaded correctly.

In [ ]:
def verify_graph_structure(driver):
    """Query the graph to verify the structure."""
    with driver.session() as session:
        # Count nodes
        result = session.run("""
            MATCH (c:Company) 
            RETURN count(c) AS company_count
        """)
        company_count = result.single()["company_count"]
        
        result = session.run("""
            MATCH (a:Address) 
            RETURN count(a) AS address_count
        """)
        address_count = result.single()["address_count"]
        
        result = session.run("""
            MATCH (s:SLA) 
            RETURN count(s) AS sla_count
        """)
        sla_count = result.single()["sla_count"]
        
        # Sample query
        result = session.run("""
            MATCH (c:Company)-[:LOCATED_AT]->(a:Address)
            MATCH (c)-[:HAS_SLA]->(s:SLA)
            RETURN c.name AS company, 
                   a.country AS country, 
                   c.spend_2024 AS spend,
                   s.title AS sla_title
            LIMIT 5
        """)
        samples = [record.data() for record in result]
        
        print(f"\n📊 Graph Statistics:")
        print(f"  Companies: {company_count}")
        print(f"  Addresses: {address_count}")
        print(f"  SLAs: {sla_count}")
        print(f"\n📋 Sample Records:")
        for i, sample in enumerate(samples, 1):
            print(f"  {i}. {sample['company']} ({sample['country']}) - {sample['sla_title']}")
            print(f"     Spend: €{sample['spend']:,.2f}" if sample['spend'] else "     Spend: N/A")

# Execute verification
verify_graph_structure(driver)

## 9. Cleanup

Close the Neo4j driver connection.

In [ ]:
driver.close()
print("✓ Connection closed")

---
## Parte 3: Consultas Cypher

Aprendemos a linguagem de consulta Cypher do Neo4j para navegar e extrair
informacoes do grafo de conhecimento.

## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [ ]:
%pip install python-dotenv neo4j

## 1. Setup and Imports

Import required libraries and load environment variables.

In [ ]:
from __future__ import annotations
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Carregar environment variables from .env file
load_dotenv()

## 2. Connect to Neo4j

Create a reusable driver function that reads connection details from environment variables.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
```

In [ ]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    pwd = os.getenv("NEO4J_PASSWORD")
    if not uri or not user or not pwd:
        raise RuntimeError("Missing Neo4j env vars")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Testar connection
driver = get_driver()
print("✓ Connected to Neo4j")

## 3. List Clauses for a Specific SLA

Retrieve all clauses for one SLA, ordered by their original position in the document.

In [ ]:
def list_clauses_for_sla(sla_id: str) -> List[Dict[str, Any]]:
    """Return all clauses for one SLA ordered by their original position."""
    cypher = """
    MATCH (s:SLA {id: $sla_id})-[:HAS_CLAUSE]->(cl:Clause)
    RETURN cl.order AS order,
           cl.title AS title,
           cl.text AS text
    ORDER BY order
    """
    driver = get_driver()
    with driver.session() as session:
        response = session.run(cypher, sla_id=sla_id)
        records = [r.data() for r in response]
        return records

# Execute
records_clauses = list_clauses_for_sla(sla_id="SLA1")
print(f"✓ Found {len(records_clauses)} clauses for SLA1")
for i, record in enumerate(records_clauses[:3], 1):
    print(f"  {i}. Order {record['order']}: {record['title']}")

## 5. Find Clauses of a Specific Type

List all clauses of a specific type (e.g., "Termination") across all suppliers and SLAs.

In [ ]:
def clauses_of_type(clause_type: str) -> List[Dict[str, Any]]:
    """List all clauses of a specific ClauseType across suppliers."""
    cypher = """
    MATCH (c:Company)-[:HAS_SLA]->(s:SLA)-[:HAS_CLAUSE]->(cl:Clause)
    MATCH (cl)-[:OF_TYPE]->(t:ClauseType {name: $clause_type})
    RETURN c.name AS company,
           s.id AS sla_id,
           cl.order AS clause_order,
           cl.title AS clause_title,
           cl.text AS clause_text
    ORDER BY company, sla_id, clause_order
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, clause_type=clause_type)]

# Execute
records_clause_types = clauses_of_type(clause_type="Termination")
print(f"✓ Found {len(records_clause_types)} Termination clauses")
for i, record in enumerate(records_clause_types[:3], 1):
    print(f"  {i}. {record['company']} - {record['sla_id']}: {record['clause_title']}")

## 6. High-Spend Companies Missing Termination Clauses

Identify companies with high spend whose SLAs lack termination clauses - a potential risk indicator.

In [ ]:
def high_spend_missing_termination(min_spend: float) -> List[Dict[str, Any]]:
    """Return companies above min_spend whose SLAs lack a termination clause."""
    cypher = """
    MATCH (c:Company)-[:HAS_SLA]->(s:SLA)
    WHERE c.spend_2024 > $min_spend
    OPTIONAL MATCH (s)-[:HAS_CLAUSE]->(cl:Clause)-[:OF_TYPE]->(t:ClauseType {name: "Termination"})
    WITH c, s, count(cl) AS num_termination
    WHERE num_termination = 0
    RETURN c.name AS company,
           c.spend_2024 AS spend_2024,
           s.id AS sla_id
    ORDER BY spend_2024 DESC
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, min_spend=min_spend)]

# Execute
records_min_spend = high_spend_missing_termination(min_spend=500000)
print(f"✓ Found {len(records_min_spend)} high-spend companies without termination clauses")
for i, record in enumerate(records_min_spend[:5], 1):
    spend = record['spend_2024']
    print(f"  {i}. {record['company']} - €{spend:,.2f}" if spend else f"  {i}. {record['company']}")

## 7. Search Availability Clauses

Inspect availability clauses across suppliers, searching for specific SLA commitments (e.g., "99.9%" uptime).

In [ ]:
def availability_clauses(search_phrase: str) -> List[Dict[str, Any]]:
    """Inspect availability clauses per supplier."""
    cypher = """
    MATCH (c:Company)-[:HAS_SLA]->(s:SLA)-[:HAS_CLAUSE]->(cl:Clause)
    MATCH (cl)-[:OF_TYPE]->(t:ClauseType {name: "Availability"})
    WHERE toLower(cl.text) CONTAINS toLower($phrase)
    RETURN c.name AS company,
           s.id AS sla_id,
           cl.order AS clause_order,
           cl.text AS availability_text
    ORDER BY company, sla_id, clause_order
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, phrase=search_phrase)]

# Execute
availability_clauses_records = availability_clauses(search_phrase="99.9")
print(f"✓ Found {len(availability_clauses_records)} availability clauses containing '99.9'")
for i, record in enumerate(availability_clauses_records[:3], 1):
    print(f"  {i}. {record['company']} - {record['sla_id']}")
    text_preview = record['availability_text'][:100] + "..." if len(record['availability_text']) > 100 else record['availability_text']
    print(f"     {text_preview}")

## 8. EU Data Protection Clauses by Country

Retrieve data protection clauses for suppliers located in specific EU countries, useful for GDPR compliance reviews.

In [ ]:
def eu_data_protection_clauses(countries: list[str]):
    """Retrieve data protection clauses for suppliers in given EU countries."""
    cypher = """
    MATCH (c:Company)-[:LOCATED_AT]->(a:Address)
    WHERE a.country IN $countries
    MATCH (c)-[:HAS_SLA]->(s:SLA)-[:HAS_CLAUSE]->(cl:Clause)
    MATCH (cl)-[:OF_TYPE]->(t:ClauseType {name: "DataProtection"})
    RETURN c.name AS company,
           a.country AS country,
           s.id AS sla_id,
           cl.order AS clause_order,
           cl.text AS data_protection_clause
    ORDER BY country, company, sla_id, clause_order
    """
    driver = get_driver()
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, countries=countries)]

countries = ["Germany", "France", "Netherlands"]
eu_data_protection_clauses_records = eu_data_protection_clauses(countries=countries)
print(f"✓ Found {len(eu_data_protection_clauses_records)} data protection clauses in {', '.join(countries)}")
for i, record in enumerate(eu_data_protection_clauses_records[:3], 1):
    print(f"  {i}. {record['company']} ({record['country']}) - {record['sla_id']}")

## 9. Cleanup

Close the Neo4j driver connection.

In [ ]:
driver.close()
print("✓ Connection closed")

---
## Parte 4: Embeddings e Busca Vetorial no Grafo

Combinamos a estrutura do grafo com embeddings vetoriais para realizar
busca semantica diretamente sobre os nos do Neo4j.

## 0. Install Required Packages

First, install all the necessary packages for this notebook.

In [ ]:
%pip install python-dotenv==1.0.0 neo4j==5.0.0 openai==1.12.0 pandas==2.0.0

## 1. Setup and Imports

Import required libraries and load environment variables.

In [ ]:
from __future__ import annotations
import os
from typing import List, Dict, Any
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

# Carregar environment variables from .env file
load_dotenv()
client = OpenAI()

## 2. Connect to Neo4j

Create a reusable driver function that reads connection details from environment variables.

Make sure your `.env` file contains:
```
NEO4J_URI=neo4j://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
OPENAI_API_KEY=your_openai_api_key
```

In [ ]:
def get_driver():
    """Create Neo4j driver instance from environment variables."""
    uri = os.getenv("NEO4J_URI")
    user = os.getenv("NEO4J_USERNAME")
    pwd = os.getenv("NEO4J_PASSWORD")
    return GraphDatabase.driver(uri, auth=(user, pwd))

# Testar connection
driver = get_driver()
print("✓ Connected to Neo4j")
driver.close()

## 3. Create Embeddings for Clauses

Generate embeddings for all clause text using OpenAI's text-embedding-3-small model and store them in the graph.

In [ ]:
def create_embedding(text: str):
    """Generate embeddings using OpenAI's text-embedding-3-small model."""
    return (
        client.embeddings.create(model="text-embedding-3-small", input=text)
        .data[0]
        .embedding
    )

def create_clause_embeddings():
    """Add embeddings to all Clause nodes in the graph."""
    driver = get_driver()
    with driver.session() as session:
        result = session.run("MATCH (cl:Clause) RETURN cl.id AS id, cl.text AS text")
        clauses = list(result)
        print(f"Creating embeddings for {len(clauses)} clauses...")
        
        for i, row in enumerate(clauses, 1):
            emb = create_embedding(row["text"])
            session.run(
                "MATCH (cl:Clause {id:$id}) SET cl.embedding=$emb",
                id=row["id"],
                emb=emb,
            )
            if i % 10 == 0:
                print(f"  Processed {i}/{len(clauses)} clauses")
    driver.close()
    print("✓ All clause embeddings created")

# Execute - This may take a few minutes depending on the number of clauses
create_clause_embeddings()

## 4. Create Vector Index

Create a vector index for efficient approximate nearest neighbor (ANN) search on clause embeddings.

In [ ]:
def create_vector_index():
    """Create a vector index for semantic search on Clause embeddings."""
    driver = get_driver()
    with driver.session() as session:
        session.run(
            """
            CREATE VECTOR INDEX clause_embeddings IF NOT EXISTS
            FOR (c:Clause) ON c.embedding
            OPTIONS { 
                indexConfig: { 
                    `vector.dimensions`: 1536, 
                    `vector.similarity_function`: "cosine" 
                } 
            }
            """
        )
    driver.close()
    print("✓ Vector index created")

# Execute
create_vector_index()

## 5. Semantic Search

Find clauses semantically similar to a natural language query using vector similarity search.

In [ ]:
def semantic_search(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Find clauses semantically similar to the query."""
    driver = get_driver()
    emb = create_embedding(query)
    cypher = """
    CALL db.index.vector.queryNodes(
        "clause_embeddings", $top_k, $embedding
    )
    YIELD node, score
    RETURN node.title AS title, 
           node.text AS text, 
           score 
    ORDER BY score DESC
    """
    with driver.session() as session:
        rows = [r.data() for r in session.run(cypher, top_k=top_k, embedding=emb)]
    driver.close()
    return rows

# Execute
query = "What is the uptime guarantee?"
results = semantic_search(query, top_k=3)
print(f"✓ Found {len(results)} semantically similar clauses for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['title']} (score: {result['score']:.4f})")
    text_preview = result['text'][:150] + "..." if len(result['text']) > 150 else result['text']
    print(f"   {text_preview}\n")

## 6. Hybrid Search by Industry

Combine semantic similarity search with graph-based filtering to find relevant clauses within a specific industry.

In [ ]:
def hybrid_search_by_industry(
    query: str, industry: str, top_k: int = 5
) -> List[Dict[str, Any]]:
    """Combine semantic search with industry filtering."""
    driver = get_driver()
    emb = create_embedding(query)
    cypher = """
    CALL db.index.vector.queryNodes(
        "clause_embeddings", $top_k, $embedding
    ) 
    YIELD node, score
    MATCH (node)<-[:HAS_CLAUSE]-(s:SLA)<-[:HAS_SLA]-(c:Company)
    WHERE c.industry = $industry
    RETURN c.name AS company, 
           s.id AS sla_id, 
           node.title AS clause_title, 
           node.text AS clause_text, 
           score
    ORDER BY score DESC
    """
    with driver.session() as session:
        rows = [
            r.data()
            for r in session.run(cypher, top_k=top_k, embedding=emb, industry=industry)
        ]
    driver.close()
    return rows

# Execute
query = "What are the data privacy obligations?"
industry = "Healthcare"
results = hybrid_search_by_industry(query=query, industry=industry, top_k=3)
print(f"✓ Found {len(results)} relevant clauses in {industry} industry for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['company']} - {result['clause_title']} (score: {result['score']:.4f})")
    text_preview = result['clause_text'][:150] + "..." if len(result['clause_text']) > 150 else result['clause_text']
    print(f"   {text_preview}\n")